# Run Day 4 on Kaggle

Attach a private dataset named `octlm-code` that contains this repository's files at its root. In Session options, select **GPU T4 x2** and turn on **Internet** for the first run. Then choose **Save Version > Save & Run All**. The notebook uses one T4.

The first run downloads and encodes TinyStories before training. To reuse earlier token files or a saved checkpoint, attach that output as another input and set `PREVIOUS` below to its folder containing `data/train.bin`, `data/valid.bin`, `bpe.json`, and optionally `run.pt`. Kaggle input is read-only, so the notebook copies those files to `/kaggle/working/day4`.


In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

import torch


INPUT = Path("/kaggle/input")


def input_dir(name: str) -> Path:
    # A dataset attached in the UI mounts at input/<name>; one attached through the API
    # mounts at input/<owner>/<name>.
    found = sorted(INPUT.glob(f"*/{name}")) + sorted(INPUT.glob(f"*/*/{name}"))
    return found[0] if found else INPUT / name


CODE = input_dir("octlm-code")
PREVIOUS = None  # Example: input_dir("octlm-day4") / "day4"
OUT = Path("/kaggle/working/day4")
DATA = OUT / "data"
TOKENIZER = OUT / "bpe.json"
CHECKPOINT = OUT / "run.pt"
METRICS = OUT / "metrics.jsonl"
SAMPLES = OUT / "samples.jsonl"

if not (CODE / "octlm/day4.py").is_file():
    raise FileNotFoundError(f"Attach the octlm-code dataset with octlm/day4.py at {CODE}")
if not torch.cuda.is_available() or torch.cuda.get_device_capability(0) != (7, 5):
    raise RuntimeError("Select GPU T4 x2 in Kaggle Session options")
print("PyTorch:", torch.__version__, "GPU:", torch.cuda.get_device_name(0), flush=True)

OUT.mkdir(parents=True, exist_ok=True)
if PREVIOUS is not None:
    for name in ("data/train.bin", "data/valid.bin", "bpe.json", "run.pt", "metrics.jsonl"):
        source = PREVIOUS / name
        if source.is_file():
            target = OUT / name
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, target)


def run(*args):
    subprocess.run([sys.executable, "-m", "octlm.day4", *map(str, args)], cwd=CODE, check=True)

In [ ]:
if (
    not (DATA / "train.bin").is_file()
    or not (DATA / "valid.bin").is_file()
    or not TOKENIZER.is_file()
):
    print("Preparing TinyStories", flush=True)
    run("prepare", "--data", DATA, "--tokenizer", TOKENIZER)
print("Train tokens:", (DATA / "train.bin").stat().st_size // 2, flush=True)

In [ ]:
run(
    "train",
    "--data",
    DATA,
    "--tokenizer",
    TOKENIZER,
    "--checkpoint",
    CHECKPOINT,
    "--metrics",
    METRICS,
    "--samples-out",
    SAMPLES,
    "--device",
    "cuda:0",
)
run(
    "samples",
    "--tokenizer",
    TOKENIZER,
    "--checkpoint",
    CHECKPOINT,
    "--samples-out",
    SAMPLES,
    "--temperature",
    0,
    "--device",
    "cuda:0",
)
print("Final record:", METRICS.read_text().splitlines()[-1], flush=True)
print("Saved checkpoint:", CHECKPOINT, flush=True)